# ToolNode 를 사용하여 도구를 호출하는 방법

이번 튜토리얼에서는 도구 호출을 위한 LangGraph의 사전 구축(pre-built) 컴포넌트인 `ToolNode` 사용 방법을 다룹니다.

`ToolNode`는 메시지 목록이 포함된 그래프 상태를 입력으로 받아 도구 호출 결과로 상태를 업데이트하는 LangChain Runnable입니다. 이는 LangGraph의 사전 구축 에이전트와 즉시 사용할 수 있도록 설계되었으며, 상태에 적절한 리듀서가 있는 `messages` 키가 포함된 경우 모든 `StateGraph` 와 함께 작동할 수 있습니다.

> 참고 문서: [LangChain Tools - ToolNode](https://docs.langchain.com/oss/python/langchain/tools#toolnode)

## 환경 설정

튜토리얼을 시작하기 전에 필요한 환경을 설정합니다. `dotenv`로 API 키를 로드하고, LangSmith 추적에 필요한 환경 변수를 직접 설정합니다.

아래 코드에서는 환경 변수를 로드하고 LangSmith 프로젝트를 설정합니다.

In [1]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv("../.env", override=True)

True

In [ ]:
# LangSmith 추적을 설정합니다. https://smith.langchain.com
import os

# 프로젝트 이름을 입력합니다.
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_TRACING_V2"] = "false"
os.environ["LANGSMITH_PROJECT"] = "LangGraph-V1-Tutorial"

## 도구 정의

`ToolNode`에서 사용할 도구들을 정의합니다. LangChain의 `@tool` 데코레이터를 사용하면 일반 Python 함수를 도구로 변환할 수 있습니다. 각 도구에는 docstring으로 설명을 작성하여 LLM이 언제 해당 도구를 사용할지 판단할 수 있도록 합니다.

아래 코드에서는 뉴스 검색 도구와 Python 코드 실행 도구를 정의합니다. `TAVILY_API_KEY`가 있으면 Tavily 뉴스 검색을 사용하고, 없으면 Google News RSS를 사용합니다.

In [ ]:
import os
import xml.etree.ElementTree as ET
from typing import Any, Dict, List
from urllib.parse import quote_plus

import requests
from langchain_core.tools import tool
from langchain_experimental.tools import PythonREPLTool
from langchain_tavily import TavilySearch

python_repl_tool = PythonREPLTool()


def _normalize_tavily_results(response: Any) -> List[Dict[str, Any]]:
    results = response.get("results", response) if isinstance(response, dict) else response
    if not isinstance(results, list):
        return [{"content": str(results)}]

    return [
        {
            "title": item.get("title"),
            "url": item.get("url"),
            "content": item.get("content"),
        }
        for item in results
        if isinstance(item, dict)
    ]


def _search_google_news_rss(query: str, k: int = 5) -> List[Dict[str, str]]:
    encoded_query = quote_plus(query)
    url = f"https://news.google.com/rss/search?q={encoded_query}&hl=ko&gl=KR&ceid=KR:ko"
    response = requests.get(url, timeout=10)
    response.raise_for_status()

    root = ET.fromstring(response.content)
    items = root.findall("./channel/item")[:k]
    return [
        {
            "title": item.findtext("title", default=""),
            "url": item.findtext("link", default=""),
            "content": item.findtext("description", default=""),
        }
        for item in items
    ]


# 도구 생성
@tool
def search_news(query: str) -> List[Dict[str, Any]]:
    """뉴스 키워드로 검색합니다."""
    try:
        if os.getenv("TAVILY_API_KEY"):
            tavily_search_tool = TavilySearch(max_results=5, topic="news")
            return _normalize_tavily_results(tavily_search_tool.invoke({"query": query}))

        return _search_google_news_rss(query, k=5)
    except Exception as exc:
        return [{"content": f"뉴스 검색 실패: {exc}"}]


@tool
def python_code_interpreter(code: str):
    """Python 코드를 실행합니다."""
    return python_repl_tool.invoke(code)

### ToolNode 초기화

정의한 도구들을 리스트로 묶어 `ToolNode`에 전달합니다. `ToolNode`는 이 도구 리스트를 받아 메시지의 `tool_calls` 정보에 따라 적절한 도구를 실행하는 역할을 합니다.

아래 코드에서는 도구 리스트를 생성하고 `ToolNode`를 초기화합니다.

In [4]:
from langgraph.prebuilt import ToolNode, tools_condition

# 도구 리스트 생성
tools = [search_news, python_code_interpreter]

# ToolNode 초기화
tool_node = ToolNode(tools)

## ToolNode 수동 호출

`ToolNode`를 직접 호출하여 도구 실행을 테스트할 수 있습니다. 이 방식은 그래프 없이 도구의 동작을 빠르게 확인하거나 디버깅할 때 유용합니다.

수동 호출 테스트를 위해 `ToolNode`만 포함하는 간단한 테스트 그래프를 생성하여 사용합니다.

`ToolNode`는 메시지 목록과 함께 그래프 상태에서 작동합니다. 

- **중요**: 이때 목록의 마지막 메시지는 `tool_calls` 속성을 포함하는 `AIMessage`여야 합니다.

아래 코드에서는 `ToolNode`만 포함하는 최소한의 테스트 그래프를 생성하고, 단일 도구 호출을 포함하는 `AIMessage`를 전달하여 도구를 실행합니다.

In [5]:
from langchain_core.messages import AIMessage
from langgraph.graph import StateGraph, MessagesState, START, END

# 단일 도구 호출을 포함하는 AI 메시지 객체 생성
# AIMessage 객체이어야 함
message_with_single_tool_call = AIMessage(
    content="",
    tool_calls=[
        {
            "name": "search_news",  # 도구 이름
            "args": {"query": "AI"},  # 도구 인자
            "id": "tool_call_id",  # 도구 호출 ID
            "type": "tool_call",  # 도구 호출 유형
        }
    ],
)

# ToolNode 테스트를 위한 간단한 그래프 생성
test_graph = StateGraph(MessagesState)
test_graph.add_node("tools", tool_node)
test_graph.add_edge(START, "tools")
test_graph.add_edge("tools", END)
test_app = test_graph.compile()

# 테스트 그래프를 통한 메시지 처리 및 뉴스 검색 실행
test_app.invoke({"messages": [message_with_single_tool_call]})

{'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={}, id='c9aafd0f-805e-4db3-b504-6a3aafe01203', tool_calls=[{'name': 'search_news', 'args': {'query': 'AI'}, 'id': 'tool_call_id', 'type': 'tool_call'}], invalid_tool_calls=[]),
  ToolMessage(content='[{"url": "https://news.google.com/rss/articles/CBMiVkFVX3lxTE0tZ1lBaUd6UFhRNHlJemZJejM2d3RGcldVV2hWV24xbUZSZGl5b2VsUU1xRmdpSS1RMndISnpKWGpSU3h0ZjNQYW55Szk2OGhNQU5uYXdR?oc=5", "content": "현대차그룹, 새만금 로봇·AI·수소 거점에 총 9조원 투자…\\"미래기업 도약\\" - 지디넷코리아"}, {"url": "https://news.google.com/rss/articles/CBMiWkFVX3lxTE81MXdsX2JOaUtfczNiZG1MNGZ6UFViTkI0RWdZbjdNS2RQRm93ZktjSFNpbWJhZzF2eWVUZEc3dnlXRWhTaTNrc2FVcFJ3ZHl1RFpSZGZqUWJyQdIBX0FVX3lxTE1NVkx0NGRwa29TLVZkS2xDcWhjUEdyM3lrUXlOa0FlTVR5VWVoV21QamducV9PVGhFalgyM3VMckxLcGRFM21NR2g0eWFqLXRyS0JjUFl3by1BVkM1T1N3?oc=5", "content": "현대차그룹, 새만금에 9조원 투자···로봇·수소·AI 거점 조성 - 경향신문"}, {"url": "https://news.google.com/rss/articles/CBMiT0FVX3lxTFBPdno1OF9iSGN2ODZvWUsyS1hyajZWUGRHSURWMjRVUG5KS3Ay

일반적으로 `AIMessage`를 수동으로 생성할 필요가 없으며, 도구 호출을 지원하는 모든 LangChain 채팅 모델에서 자동으로 생성됩니다.

또한 `AIMessage`의 `tool_calls` 매개변수에 여러 도구 호출을 전달하면 `ToolNode`를 사용하여 병렬 도구 호출을 수행할 수 있습니다.

아래 코드에서는 앞서 생성한 테스트 그래프를 활용하여 뉴스 검색과 Python 코드 실행을 동시에 호출하는 다중 도구 호출 예시를 보여줍니다.

In [6]:
# 다중 도구 호출을 포함하는 AI 메시지 객체 생성 및 초기화
message_with_multiple_tool_calls = AIMessage(
    content="",
    tool_calls=[
        {
            "name": "search_news",
            "args": {"query": "AI"},
            "id": "tool_call_id_1",
            "type": "tool_call",
        },
        {
            "name": "python_code_interpreter",
            "args": {"code": "print('Hello, World!')"},
            "id": "tool_call_id_2",
            "type": "tool_call",
        },
    ],
)

# 테스트 그래프를 통한 다중 도구 호출 실행
test_app.invoke({"messages": [message_with_multiple_tool_calls]})

Python REPL can execute arbitrary code. Use with caution.


{'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={}, id='a805e6dd-c2aa-46b8-bf2c-5052a1b2628d', tool_calls=[{'name': 'search_news', 'args': {'query': 'AI'}, 'id': 'tool_call_id_1', 'type': 'tool_call'}, {'name': 'python_code_interpreter', 'args': {'code': "print('Hello, World!')"}, 'id': 'tool_call_id_2', 'type': 'tool_call'}], invalid_tool_calls=[]),
  ToolMessage(content='[{"url": "https://news.google.com/rss/articles/CBMiVkFVX3lxTE0tZ1lBaUd6UFhRNHlJemZJejM2d3RGcldVV2hWV24xbUZSZGl5b2VsUU1xRmdpSS1RMndISnpKWGpSU3h0ZjNQYW55Szk2OGhNQU5uYXdR?oc=5", "content": "현대차그룹, 새만금 로봇·AI·수소 거점에 총 9조원 투자…\\"미래기업 도약\\" - 지디넷코리아"}, {"url": "https://news.google.com/rss/articles/CBMiWkFVX3lxTE81MXdsX2JOaUtfczNiZG1MNGZ6UFViTkI0RWdZbjdNS2RQRm93ZktjSFNpbWJhZzF2eWVUZEc3dnlXRWhTaTNrc2FVcFJ3ZHl1RFpSZGZqUWJyQdIBX0FVX3lxTE1NVkx0NGRwa29TLVZkS2xDcWhjUEdyM3lrUXlOa0FlTVR5VWVoV21QamducV9PVGhFalgyM3VMckxLcGRFM21NR2g0eWFqLXRyS0JjUFl3by1BVkM1T1N3?oc=5", "content": "현대차그룹, 새만금에 9조원 투자···로봇·수소·A

## LLM과 함께 사용하기

실제 애플리케이션에서는 `AIMessage`를 수동으로 생성하지 않고, LLM이 자동으로 생성한 `tool_calls`를 사용합니다. LLM이 도구를 호출할 수 있도록 하려면 먼저 `bind_tools()` 메서드로 사용 가능한 도구들을 알려주어야 합니다.

도구 호출 기능이 있는 채팅 모델을 사용하기 위해서는 먼저 모델이 사용 가능한 도구들을 인식하도록 해야 합니다. 이는 `init_chat_model` 함수로 모델을 초기화한 뒤, `.bind_tools` 메서드를 호출하여 수행합니다.

아래 코드에서는 `init_chat_model`로 모델을 초기화하고, 앞서 정의한 도구들을 바인딩합니다.

In [7]:
from langchain.chat_models import init_chat_model

# LLM 모델 초기화 및 도구 바인딩
# OpenAI 키를 사용하는 경우 gpt-5.2, gpt-4.1-mini 등으로 변경하세요
model_with_tools = init_chat_model("claude-sonnet-4-5").bind_tools(tools)

In [8]:
# LLM 모델의 도구 호출 응답 생성
ai_message = model_with_tools.invoke("처음 5개의 소수를 출력하는 python code 를 작성해줘")

# 도구 호출 확인
ai_message.tool_calls

[{'name': 'python_code_interpreter',
  'args': {'code': '\n# 처음 5개의 소수를 찾는 프로그램\n\ndef is_prime(n):\n    """숫자가 소수인지 확인하는 함수"""\n    if n < 2:\n        return False\n    for i in range(2, int(n ** 0.5) + 1):\n        if n % i == 0:\n            return False\n    return True\n\n# 처음 5개의 소수 찾기\nprimes = []\nnum = 2\n\nwhile len(primes) < 5:\n    if is_prime(num):\n        primes.append(num)\n    num += 1\n\nprint("처음 5개의 소수:")\nfor prime in primes:\n    print(prime)\n'},
  'id': 'toolu_01YKt6B4aFrsVhRzeMWviqdB',
  'type': 'tool_call'}]

채팅 모델이 생성한 AI 메시지에는 이미 `tool_calls`가 채워져 있으므로, 이를 테스트 그래프를 통해 `ToolNode`에 전달할 수 있습니다. 이 방식은 LLM의 도구 호출 결정과 실제 도구 실행을 분리하여 처리하는 패턴으로, 에이전트 그래프에서 핵심적으로 사용됩니다.

아래 코드에서는 LLM이 생성한 도구 호출 메시지를 테스트 그래프에 전달하여 실행 결과를 확인합니다.

In [9]:
# 테스트 그래프를 통한 메시지 처리 및 LLM 모델의 도구 기반 응답 생성
test_app.invoke({"messages": [ai_message]})

{'messages': [AIMessage(content=[{'id': 'toolu_01YKt6B4aFrsVhRzeMWviqdB', 'input': {'code': '\n# 처음 5개의 소수를 찾는 프로그램\n\ndef is_prime(n):\n    """숫자가 소수인지 확인하는 함수"""\n    if n < 2:\n        return False\n    for i in range(2, int(n ** 0.5) + 1):\n        if n % i == 0:\n            return False\n    return True\n\n# 처음 5개의 소수 찾기\nprimes = []\nnum = 2\n\nwhile len(primes) < 5:\n    if is_prime(num):\n        primes.append(num)\n    num += 1\n\nprint("처음 5개의 소수:")\nfor prime in primes:\n    print(prime)\n'}, 'name': 'python_code_interpreter', 'type': 'tool_use', 'caller': {'type': 'direct'}}], additional_kwargs={}, response_metadata={'id': 'msg_018cF2CzMFxfrW2SzAVioXDt', 'model': 'claude-sonnet-4-5-20250929', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'input_tokens': 734, 'output_tokens': 245, 'server_tool_use': None, 'service_

## Agent 그래프와 함께 사용하기

`ToolNode`의 가장 일반적인 사용 사례는 LangGraph 에이전트 그래프 내에서 도구 실행 노드로 활용하는 것입니다. 에이전트는 사용자 질문을 받아 필요에 따라 도구를 호출하고, 그 결과를 바탕으로 최종 응답을 생성합니다.

### 에이전트 그래프 구현

다음으로, StateGraph를 사용하여 에이전트 그래프를 구현합니다. 이 에이전트는 쿼리를 입력으로 받아, 필요한 정보를 얻을 때까지 반복적으로 도구들을 호출합니다. `tools_condition`은 LLM 응답에 도구 호출이 있는지 확인하여 다음 노드를 결정하는 사전 구축된 조건 함수입니다.

아래 코드에서는 에이전트 그래프를 정의하고 컴파일합니다.

In [10]:
# LangGraph 워크플로우 상태 및 메시지 처리를 위한 타입 임포트
from langgraph.graph import StateGraph, MessagesState, START, END


# LLM 모델을 사용하여 메시지 처리 및 응답 생성, 도구 호출이 포함된 응답 반환
def call_model(state: MessagesState):
    messages = state["messages"]
    response = model_with_tools.invoke(messages)
    return {"messages": [response]}


# 메시지 상태 기반 워크플로우 그래프 초기화
workflow = StateGraph(MessagesState)

# 에이전트와 도구 노드 정의 및 워크플로우 그래프에 추가
workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)

# 워크플로우 시작점에서 에이전트 노드로 연결
workflow.add_edge(START, "agent")

# 에이전트 노드에서 조건부 분기 설정, 도구 노드 또는 종료 지점으로 연결
workflow.add_conditional_edges("agent", tools_condition)

# 도구 노드에서 에이전트 노드로 순환 연결
workflow.add_edge("tools", "agent")

# 정의된 워크플로우 그래프 컴파일 및 실행 가능한 애플리케이션 생성
app = workflow.compile()

In [ ]:
from IPython.display import Image, display


def visualize_graph(graph):
    display(Image(graph.get_graph().draw_mermaid_png()))


visualize_graph(app)

### 에이전트 실행 테스트

구현한 에이전트를 다양한 질문으로 테스트해봅니다. Python 코드 작성 요청, 뉴스 검색 요청, 그리고 도구 호출이 필요 없는 일반 대화까지 에이전트가 적절히 처리하는지 확인합니다.

아래 코드에서는 에이전트를 실행하고 결과를 출력합니다.

In [ ]:
def stream_graph(graph, inputs):
    for event in graph.stream(inputs, stream_mode="values"):
        event["messages"][-1].pretty_print()


# 실행 및 결과 확인
stream_graph(
    app,
    inputs={"messages": [("human", "처음 5개의 소수를 출력하는 python code 를 작성해줘")]},
)

In [13]:
# 검색 질문 수행
stream_graph(
    app,
    inputs={"messages": [("human", "search google news about AI")]},
)


🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
[{"url": "https://news.google.com/rss/articles/CBMiVkFVX3lxTE0tZ1lBaUd6UFhRNHlJemZJejM2d3RGcldVV2hWV24xbUZSZGl5b2VsUU1xRmdpSS1RMndISnpKWGpSU3h0ZjNQYW55Szk2OGhNQU5uYXdR?oc=5", "content": "현대차그룹, 새만금 로봇·AI·수소 거점에 총 9조원 투자…\"미래기업 도약\" - 지디넷코리아"}, {"url": "https://news.google.com/rss/articles/CBMiWkFVX3lxTE81MXdsX2JOaUtfczNiZG1MNGZ6UFViTkI0RWdZbjdNS2RQRm93ZktjSFNpbWJhZzF2eWVUZEc3dnlXRWhTaTNrc2FVcFJ3ZHl1RFpSZGZqUWJyQdIBX0FVX3lxTE1NVkx0NGRwa29TLVZkS2xDcWhjUEdyM3lrUXlOa0FlTVR5VWVoV21QamducV9PVGhFalgyM3VMckxLcGRFM21NR2g0eWFqLXRyS0JjUFl3by1BVkM1T1N3?oc=5", "content": "현대차그룹, 새만금에 9조원 투자···로봇·수소·AI 거점 조성 - 경향신문"}, {"url": "https://news.google.com/rss/articles/CBMiT0FVX3lxTFBPdno1OF9iSGN2ODZvWUsyS1hyajZWUGRHSURWMjRVUG5KS3AyWTNMNFNBaWVRRW01Q21GOVJwb1c3Q0FMeXY5U2pwcHpZcEE?oc=5", "content": "李대통령 \"현대車 새만금 AI·로봇 투자, 호남 경제지도 바꿀 것\" - v.daum.net"}, {"url": "https://ne

In [14]:
# 도구 호출이 필요 없는 질문 수행
stream_graph(
    app,
    inputs={"messages": [("human", "안녕? 반가워")]},
)


🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
안녕하세요! 반갑습니다! 😊

무엇을 도와드릴까요? 궁금하신 점이나 필요하신 것이 있으시면 편하게 말씀해 주세요!

## 도구 오류 처리

`ToolNode`는 도구 실행 중 발생하는 오류도 자동으로 처리합니다. 기본적으로 `handle_tool_errors=True`가 설정되어 있어, 도구에서 예외가 발생하면 오류 메시지를 `ToolMessage`로 변환하여 LLM에게 전달합니다. 이를 통해 LLM이 오류 상황을 인식하고 다른 접근 방식을 시도할 수 있습니다.

오류 처리를 비활성화하려면 `ToolNode(tools, handle_tool_errors=False)`로 설정하면 됩니다.

### handle_tool_errors 옵션

`handle_tool_errors` 매개변수는 다양한 형태로 설정할 수 있습니다. 상황에 맞는 에러 처리 전략을 선택하면 LLM이 오류를 더 효과적으로 처리할 수 있습니다.

| 설정 값 | 동작 |
|---------|------|
| `True` (기본값) | 모든 오류를 잡아 오류 메시지를 LLM에 전달합니다 |
| `False` | 오류 처리를 비활성화하여 예외를 그대로 발생시킵니다 |
| `"커스텀 메시지"` | 오류 발생 시 지정된 문자열을 LLM에 전달합니다 |
| `callable` | 오류 객체를 받아 커스텀 메시지를 반환하는 함수를 사용합니다 |
| `(ValueError, TypeError)` | 지정된 예외 타입만 처리합니다 |

아래 코드에서는 각 옵션의 사용 방법을 보여줍니다.

In [15]:
# 1. 기본 오류 처리 (True) - 모든 오류를 잡아 ToolMessage로 변환
tool_node_default = ToolNode(tools, handle_tool_errors=True)

# 2. 오류 처리 비활성화 (False) - 예외가 그대로 발생
tool_node_no_handle = ToolNode(tools, handle_tool_errors=False)

# 3. 커스텀 메시지 (String) - 고정된 오류 메시지 반환
tool_node_custom_msg = ToolNode(
    tools,
    handle_tool_errors="도구 실행 중 오류가 발생했습니다. 다른 방법을 시도해 주세요.",
)

# 4. 커스텀 핸들러 (Callable) - 오류에 따라 동적으로 메시지 생성
def custom_error_handler(error: Exception) -> str:
    """오류 타입에 따라 다른 메시지를 반환하는 핸들러"""
    if isinstance(error, ValueError):
        return f"입력값 오류: {error}"
    return f"예상치 못한 오류: {error}"


tool_node_callable = ToolNode(tools, handle_tool_errors=custom_error_handler)

# 5. 특정 예외만 처리 (Tuple) - 지정된 예외 타입만 잡고 나머지는 발생
tool_node_specific = ToolNode(tools, handle_tool_errors=(ValueError, TypeError))

### 오류 처리 동작 테스트

오류가 발생하는 도구를 정의하여 `handle_tool_errors` 옵션의 동작을 확인합니다. 오류가 발생하면 `ToolMessage`의 `status` 필드가 `"error"`로 설정되어 LLM에 전달됩니다.

아래 코드에서는 의도적으로 오류를 발생시키는 도구를 만들고, 커스텀 메시지 옵션을 적용한 `ToolNode`를 테스트 그래프에 포함하여 오류 처리 동작을 테스트합니다.

In [16]:
@tool
def error_tool(input: str) -> str:
    """항상 오류를 발생시키는 테스트 도구"""
    raise ValueError(f"잘못된 입력: {input}")


# 커스텀 메시지로 오류 처리하는 ToolNode 생성
error_tool_node = ToolNode(
    [error_tool],
    handle_tool_errors="도구 실행 중 오류가 발생했습니다. 다른 방법을 시도해 주세요.",
)

# 오류 처리 테스트를 위한 그래프 생성
error_test_graph = StateGraph(MessagesState)
error_test_graph.add_node("tools", error_tool_node)
error_test_graph.add_edge(START, "tools")
error_test_graph.add_edge("tools", END)
error_test_app = error_test_graph.compile()

# 오류가 발생하는 도구 호출 테스트
error_message = AIMessage(
    content="",
    tool_calls=[
        {
            "name": "error_tool",
            "args": {"input": "test"},
            "id": "error_test_id",
            "type": "tool_call",
        }
    ],
)

# 오류 처리 결과 확인
result = error_test_app.invoke({"messages": [error_message]})
result["messages"][-1].pretty_print()

================================= Tool Message =================================
Name: error_tool

도구 실행 중 오류가 발생했습니다. 다른 방법을 시도해 주세요.


---

## 도구에서 그래프 상태 접근하기

도구 실행 중에 그래프의 현재 상태에 접근해야 하는 경우가 있습니다. 예를 들어, 대화 기록의 길이를 확인하거나 커스텀 상태 값을 참조해야 할 때입니다. `ToolRuntime`을 사용하면 도구 함수 내에서 그래프 상태에 안전하게 접근할 수 있습니다.

`ToolRuntime`은 도구의 매개변수로 선언하면 `ToolNode`가 자동으로 주입합니다. LLM에게는 이 매개변수가 노출되지 않으므로 도구의 스키마에 영향을 주지 않습니다.

> 참고 문서: [LangChain Tools - State Injection](https://docs.langchain.com/oss/python/langchain/tools#state-injection)

아래 코드에서는 `ToolRuntime`을 사용하여 그래프 상태에 접근하는 도구를 정의합니다.

In [17]:
from langchain.tools import ToolRuntime


@tool
def get_message_count(runtime: ToolRuntime) -> str:
    """현재 대화의 메시지 수를 반환합니다."""
    messages = runtime.state["messages"]
    return f"현재 대화에 {len(messages)}개의 메시지가 있습니다."


# ToolRuntime을 사용하는 도구로 ToolNode 생성
state_tool_node = ToolNode([get_message_count])

`ToolRuntime`을 사용하는 도구를 테스트합니다. `ToolNode`가 그래프 내에서 실행될 때 자동으로 현재 그래프 상태를 주입하므로, 도구 함수 내에서 별도의 설정 없이 상태에 접근할 수 있습니다.

아래 코드에서는 테스트 그래프를 생성하고 메시지 목록과 함께 도구를 호출하여 상태 접근 결과를 확인합니다.

In [18]:
from langchain_core.messages import HumanMessage

# ToolRuntime 테스트를 위한 그래프 생성
state_test_graph = StateGraph(MessagesState)
state_test_graph.add_node("tools", state_tool_node)
state_test_graph.add_edge(START, "tools")
state_test_graph.add_edge("tools", END)
state_test_app = state_test_graph.compile()

# 상태 주입 테스트를 위한 메시지 생성
test_messages = [
    HumanMessage(content="안녕하세요"),
    AIMessage(
        content="",
        tool_calls=[
            {
                "name": "get_message_count",
                "args": {},
                "id": "state_test_id",
                "type": "tool_call",
            }
        ],
    ),
]

# 테스트 그래프를 통한 도구 호출 실행 및 상태 접근 결과 확인
stream_graph(
    state_test_app,
    inputs={"messages": test_messages},
)


🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
현재 대화에 2개의 메시지가 있습니다.

## 정리

이번 튜토리얼에서는 LangGraph의 사전 구축 컴포넌트인 `ToolNode`를 활용하여 도구를 호출하는 방법을 살펴보았습니다. 핵심 내용을 정리하면 다음과 같습니다.

- **ToolNode 기본 사용법**: `ToolNode`는 도구 리스트를 받아 초기화하며, `AIMessage`의 `tool_calls` 정보에 따라 적절한 도구를 자동으로 실행합니다.
- **단일 및 병렬 도구 호출**: `AIMessage`에 하나 또는 여러 개의 `tool_calls`를 포함하여 단일/병렬 도구 호출을 수행할 수 있습니다.
- **LLM과의 연동**: `bind_tools()` 메서드로 LLM에 도구를 바인딩하면, LLM이 자동으로 `tool_calls`가 포함된 `AIMessage`를 생성합니다.
- **에이전트 그래프 통합**: `ToolNode`와 `tools_condition`을 활용하여 LLM → 도구 호출 → 결과 반환의 반복 루프를 가진 에이전트 그래프를 구성할 수 있습니다.
- **오류 처리**: `handle_tool_errors` 옵션을 통해 도구 실행 중 발생하는 오류를 다양한 방식(`True`, 커스텀 메시지, 콜백 함수 등)으로 처리할 수 있습니다.
- **그래프 상태 접근**: `ToolRuntime`을 도구 매개변수로 선언하면, 도구 함수 내에서 그래프의 현재 상태에 접근할 수 있습니다.